# 1 Descargar Ventas y Visitas (aje-prd-analytics-artifacts-s3)

In [1]:
#!pip install 'awswrangler[redshift]'
#!pip install psycopg2-binary


!pip install --upgrade pip setuptools wheel
!pip install --upgrade pyarrow==17.0.0
!pip install "awswrangler[redshift]" --no-build-isolation
!pip install psycopg2-binary

In [2]:
import pandas as pd
import numpy as np
import redshift_connector
import awswrangler as wr
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import os
import boto3
import io
import pytz
import re
import gc

In [3]:
def comprobar_inputs():
    # Conectarse a S3
    s3 = boto3.client("s3")
    bucket_name = "aje-prd-analytics-artifacts-s3"
    prefix = "pedido_sugerido/data-v1/ecuador/"
    
    hoy = datetime.now(pytz.timezone("America/Lima")).date()
    errores = []

    # Listar objetos en la ruta de S3
    objetos = s3.list_objects_v2(Bucket=bucket_name, Prefix=prefix)

    if "Contents" not in objetos:
        print("ERROR: No se encontraron archivos en la ruta especificada.")
        return

    # Iterar sobre los objetos
    for objeto in objetos["Contents"]:
        key = objeto["Key"]
        print(key)
        # Omitir "carpetas" en S3
        if key.endswith("/"):
            continue

        last_modified = objeto["LastModified"].date()
        size_kb = objeto["Size"] / 1024  # Convertir a KB

        # Verificar si el objeto tiene contenido real (FULL_OBJECT)
        if objeto["Size"] == 0:
            errores.append(f"ERROR: El archivo {key} está vacío.")

        # Verificar si el archivo ha sido modificado hoy
        if last_modified != hoy:
            errores.append(f"ERROR: El archivo {key} no ha sido modificado hoy ({hoy}), su ultima fecha de modificacion fue {last_modified}.")

        # Verificar si el tamaño del archivo es menor a 1 KB
        if size_kb < 1:
            errores.append(f"ALERTA: El archivo {key} tiene un tamaño menor a 1 KB ({size_kb:.2f} KB).")

    # Mostrar los errores y lanzar una excepción si es necesario
    if errores:
        for error in errores:
            print(error)
        raise ValueError("Se encontraron problemas con los archivos en S3.")
    else:
        print("Todo bien :D")

In [4]:
# comprobar_inputs()

## 1.0 Descargar Maestro de productos

In [5]:
# PARA 2024
query = f"""select * from 
    comercial_ecuador.dim_producto
    where estado='A' and instancia='EC';
    """

con = wr.data_api.redshift.connect(
    cluster_id="dwh-cloud-storage-salesforce-prod",
    database="dwh_prod",
    db_user="dwhuser",
)
maestro_prod = wr.data_api.rds.read_sql_query(query, con)

In [6]:
maestro_prod.groupby(["cod_articulo_magic", "desc_articulo"]).head(1)[["cod_articulo_magic", "desc_articulo"]].to_csv("Input/EC_maestro_productos.csv", index=False)

In [7]:
# companias_test = ['6001','6002','6003','6006','6007','6008','6009',
#              '6010','6012','6013','6014','6015','6016','6017','6019',
#              '6021','6022','6023','6024','6025','E001']

# Compañia	cod_sucursal	cod_compañia_unico	Sucursal
# ECUACORCIAS S.A.	01	6001|01	MILAGRO
# BELITAMA MONTERO KARLA YOMARA	01	6002|01	BABAHOYO
############## OLGA GERARDINA YANEZ CASTILLO	01	6003|01	DURAN 
# DIPRALDET S.A	01	6006|01	IBARRA
############## CABLU S.A.S	01	6007|01	PEDRO CARBO
# NORTH-LOGISTIX S.A.S	01	6008|01	QUININDE
# DIPRALDET S.A	01	6009|01	CARCHI
############## DELGADO SALTOS HECTOR FABIAN	01	6010|01	SAMBORONDON
# LOAIZA VIZCAINO EDISON JAVIER	01	6012|01	LAGO AGRIO
# BENAVIDES PALIZ MILTON ARMANDO	01	6013|01	VENTANAS
# RAMIREZ ASANZA HIRALDA EMERITA	01	6014|01	SACHA
# MULTICOMERCIO FEZARDIEL CIA LTDA	01	6015|01	ESMERALDAS
# SANTAMARIA VALLADAREZ EDGAR MARCELO	01	6016|01	RIOBAMBA
# DISTRIBUIDORA AGUIRRE Y ORTIZ	01	6017|01	MACHACHI
# ANGEL SALOMON TRUJILLO CULQUI	01	6019|01	AMBATO FORANEO
# GASLICGROUP S.A	01	6021|01	QUEVEDO
# CORDSOL ECUADOR S.A.	01	6022|01	PENÍNSULA
# OPERADOR LOGISTICO DEL SUR CIA. LTDA.	01	6023|01	CUENCA
# CABRERA JARAMILLO MAGALY ESPERANZA	01	6024|01	LOJA CENTRAL
# ZAPATA SANCHEZ DARIO JAVIER	01	6025|01	MACHALA
# ROBERTO CARLOS REYES VIVAS	01	E001|01	CHONE



companias_test = ['6003','6010']
              
print(companias_test)

['6003', '6010']


In [8]:
len(companias_test)

2

In [9]:
clientes_ruta_test = []

In [10]:
def descargar_visitas():
    # Guardar clientes de las rutas
    global clientes_ruta_test

    # Conectarse a S3
    s3 = boto3.client("s3")
    bucket_name = "aje-prod-datalake-399723489351-us-east-2-landing-s3"
    prefix = "analytics/pedido_sugerido/sellout/data/ecuador/"

    # Listar objetos en la ruta de S3
    objetos = s3.list_objects_v2(Bucket=bucket_name, Prefix=prefix)

    print(objetos)

    # Iterar sobre los objetos
    for objeto in objetos["Contents"]:
        if objeto["Size"] > 0 and objeto["Key"].split("/")[-1] == "visitas_sellout000":
            # Obtener el nombre del objeto y descargarlo
            nombre_archivo = objeto["Key"].split("/")[-1]

            # Descargar el archivo en memoria
            response = s3.get_object(Bucket=bucket_name, Key=objeto["Key"])
            content = response["Body"].read()

            # Filtrar y guardar el archivo si corresponde
            if nombre_archivo == "visitas_sellout000":
                # Convertir los bytes a DataFrame de Pandas
                df = pd.read_csv(io.BytesIO(content), sep=";")
                df = df[
                    # (df["compania__c"] == 90)
                    (df["compania__c"].isin(companias_test))
                    # & (df.codigo_canal__c == 2)
                ].reset_index(drop=True)
                # Guardamos estos clientes para descargarlos en ventas (por si cambiaron de ruta)
                clientes_ruta_test = df["codigo_cliente__c"].unique()
                nombre_csv = f"Input/{nombre_archivo}.parquet"
                df["personalizacion"] = df["personalizacion"].astype(str)
                df.to_parquet(nombre_csv, index=False)
            else:
                continue

In [11]:
%%time
descargar_visitas()

{'ResponseMetadata': {'RequestId': 'XSXQTW27SN53NAQ3', 'HostId': '/GNutHj1BWIwo9KVrBaBoi3PkCJG1ZxfPE+hunFYn/xgUhJAzigMcfDW5xa2AQ/l5Tn+ON8Z1B4=', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amz-id-2': '/GNutHj1BWIwo9KVrBaBoi3PkCJG1ZxfPE+hunFYn/xgUhJAzigMcfDW5xa2AQ/l5Tn+ON8Z1B4=', 'x-amz-request-id': 'XSXQTW27SN53NAQ3', 'date': 'Tue, 31 Mar 2026 21:26:36 GMT', 'x-amz-bucket-region': 'us-east-2', 'content-type': 'application/xml', 'transfer-encoding': 'chunked', 'server': 'AmazonS3'}, 'RetryAttempts': 0}, 'IsTruncated': False, 'Contents': [{'Key': 'analytics/pedido_sugerido/sellout/data/ecuador/', 'LastModified': datetime.datetime(2026, 2, 24, 14, 35, 53, tzinfo=tzlocal()), 'ETag': '"d41d8cd98f00b204e9800998ecf8427e"', 'ChecksumAlgorithm': ['CRC64NVME'], 'ChecksumType': 'FULL_OBJECT', 'Size': 0, 'StorageClass': 'STANDARD'}, {'Key': 'analytics/pedido_sugerido/sellout/data/ecuador/D_stock_ec.csv', 'LastModified': datetime.datetime(2026, 3, 31, 14, 4, 30, tzinfo=tzlocal()), 'ETag': '"a82ef8a65

In [12]:
def descargar_ventas():
    # Conectarse a S3
    s3 = boto3.client("s3")
    bucket_name = "aje-prod-datalake-399723489351-us-east-2-landing-s3"
    prefix = "analytics/pedido_sugerido/sellout/data/ecuador/"

    # Listar objetos en la ruta de S3
    objetos = s3.list_objects_v2(Bucket=bucket_name, Prefix=prefix)

    print(objetos)
    
    final_csv = pd.DataFrame()

    # Iterar sobre los objetos
    for objeto in objetos["Contents"]:
        # Obtener el nombre del objeto y descargarlo
        nombre_archivo = objeto["Key"].split("/")[-1]
        if objeto["Size"] > 0 and nombre_archivo == "ventas_sellout000":
            # Descargar el archivo en memoria
            response = s3.get_object(Bucket=bucket_name, Key=objeto["Key"])
            content = response["Body"].read()
            # Convertir los bytes a DataFrame de Pandas
            df = pd.read_csv(io.BytesIO(content), sep=";")
            df["desc_formato"] = df["desc_formato"].astype(str)
            df = df[(
                    (df["cod_compania"].isin(companias_test))
                    | (df["cod_cliente"].isin(clientes_ruta_test))
                )
            ].reset_index(drop=True)
            final_csv = pd.concat([final_csv, df], ignore_index=True)
        else:
            continue

    nombre_csv = f"Input/ventas_ec_all.parquet"
    final_csv.to_parquet(nombre_csv, index=False)

In [13]:
%%time
descargar_ventas()

{'ResponseMetadata': {'RequestId': 'Y976S80EPKRGZZND', 'HostId': 'eyLVFH/bJE+7Bn9CITzU0l2/Rw5K6v7q2BNVFtBj+AlrfgesIVZ9vXdGXovNVSSoLT3QdiRHP6Rt1EVJBEzQGNAaPW0gvb4x', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amz-id-2': 'eyLVFH/bJE+7Bn9CITzU0l2/Rw5K6v7q2BNVFtBj+AlrfgesIVZ9vXdGXovNVSSoLT3QdiRHP6Rt1EVJBEzQGNAaPW0gvb4x', 'x-amz-request-id': 'Y976S80EPKRGZZND', 'date': 'Tue, 31 Mar 2026 21:26:37 GMT', 'x-amz-bucket-region': 'us-east-2', 'content-type': 'application/xml', 'transfer-encoding': 'chunked', 'server': 'AmazonS3'}, 'RetryAttempts': 0}, 'IsTruncated': False, 'Contents': [{'Key': 'analytics/pedido_sugerido/sellout/data/ecuador/', 'LastModified': datetime.datetime(2026, 2, 24, 14, 35, 53, tzinfo=tzlocal()), 'ETag': '"d41d8cd98f00b204e9800998ecf8427e"', 'ChecksumAlgorithm': ['CRC64NVME'], 'ChecksumType': 'FULL_OBJECT', 'Size': 0, 'StorageClass': 'STANDARD'}, {'Key': 'analytics/pedido_sugerido/sellout/data/ecuador/D_stock_ec.csv', 'LastModified': datetime.datetime(2026, 3, 31, 14, 4, 30

/tmp/ipykernel_30712/501304367.py:23: DtypeWarning: Columns (33) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(io.BytesIO(content), sep=";")


CPU times: user 1min 2s, sys: 10.7 s, total: 1min 13s
Wall time: 1min 39s


## 1.1 Juntar VENTAS y VISITAS

In [14]:
pan_ventas = pd.read_parquet("Input/ventas_ec_all.parquet")

In [15]:
print(sorted(pan_ventas.cod_modulo.unique()))

[0, 665, 666, 667, 668, 669, 670, 671, 672, 673, 674, 675, 676, 677, 678, 679, 680, 681, 682, 684, 685, 686, 687, 689, 690, 691, 692, 695, 698, 699, 700, 701, 702, 704, 773, 774, 775, 779, 780, 781, 782, 783, 784, 785, 786, 787, 788, 789, 790, 791, 792, 793, 8601, 80101, 80102, 80103, 80104, 80105, 80106, 80301, 80302, 80303, 80304, 80305, 80306, 80307, 80308, 80309, 80310, 80311, 80312, 80314, 80315, 80316, 80317, 80318, 80319, 80320, 80321, 80322, 80323, 80324, 80327, 80328, 80329, 80330, 80332, 80375, 80602, 80603, 80604, 80605, 80606, 80607, 80609, 80610, 80611, 80612, 80613, 80614, 80616, 80617, 80618, 80620, 80621, 80622, 80624, 80625, 80626, 80631, 80632, 80633, 80641, 80642, 80643, 80650, 80651, 80652, 80653, 80654, 80655, 80656, 80657, 80658, 80659, 80660, 80661, 80662, 80663, 80664, 80701, 80702, 80703, 80704, 80705, 80707, 80711, 80713, 80714, 80715, 80730, 80731, 80801, 80802, 80803, 80804, 80805, 80806, 80808, 80809, 80810, 80811, 80813, 80814, 80815, 80816, 80817, 80818, 

In [16]:
pan_visitas = pd.read_parquet("Input/visitas_sellout000.parquet")

In [17]:
print(sorted(pan_visitas.cod_modulo.unique()))

[80301, 80302, 80303, 80304, 80305, 80306, 80307, 80308, 80309, 80310, 80311, 80312, 80314, 80315, 80316, 80317, 80318, 80319, 80320, 80321, 80322, 80323, 80324, 80327, 80328, 80329, 80330, 80331, 80332, 81001, 81002, 81003, 81004, 81005, 81006, 81007, 81008, 81009, 81010, 81011, 81012, 81013, 81014, 81015, 81016, 81018, 81019, 81020, 81021]


In [18]:
pan_ventas.shape

(3889938, 40)

In [19]:
pan_ventas = pan_ventas[pan_ventas.cod_compania.isin(companias_test)].reset_index(drop=True)

In [20]:
pan_ventas.shape

(821818, 40)

In [21]:
pan_visitas = pan_visitas[(pan_visitas.codigo_canal__c==2)].reset_index(drop=True)

In [22]:
pan_ventas["cod_articulo_magic"] = pan_ventas["id_producto"].str.split("|").str[-1]
pan_ventas["cod_articulo_magic"] = pan_ventas["cod_articulo_magic"].astype(int)

In [23]:
pan_ventas.shape

(821818, 41)

In [24]:
# Nos quedamos solo con las filas del proceso de hoy
# Obtener la fecha de hoy en el mismo formato
hoy = int(datetime.now(pytz.timezone("America/Lima")).strftime('%Y%m%d'))
print(hoy)
# Filtrar solo las filas con la fecha de hoy
# pan_ventas = pan_ventas[pan_ventas['fecha_proceso'] == hoy]

20260331


In [25]:
pan_ventas.shape

(821818, 41)

In [26]:
# Obtener la fecha y año actual
current_date = datetime.now(pytz.timezone("America/Lima"))
# Formatear la fecha en el formato "YYYYMM"
formatted_date = current_date.strftime("%Y-%m-%d")
formatted_date

'2026-03-31'

In [27]:
# Establecer la conexión con S3
bucket_name = 'aje-analytics-ps-backup'  # nombre de bucket en S3
file_name = f'PS_Ecuador/Input/visitas_ecuador000_{formatted_date}.csv'  # nombre para el archivo en S3
s3_path = f's3://{bucket_name}/{file_name}'

# Escribir el dataframe en S3 con AWS Data Wrangler
# wr.s3.to_csv(pan_visitas, s3_path, index=False)

In [28]:
def formatear_codigo(val):
    val_str = str(val).strip()
    # Si es puramente numérico, rellena con ceros a la izquierda
    if val_str.isdigit():
        return val_str.zfill(4)
    # Si ya tiene letras (como E001), lo deja igual
    return val_str

In [29]:
# PAN VENTAS
pan_ventas["cod_compania"] = pan_ventas["cod_compania"].apply(formatear_codigo)
pan_ventas["id_cliente"] = (
    "EC"
    + "|"
    + pan_ventas["cod_compania"].astype(str)
    + "|"
    + pan_ventas["cod_cliente"].astype(str)
)

# PAN VISITAS
pan_visitas["compania__c"] = pan_visitas["compania__c"].apply(formatear_codigo)
pan_visitas["id_cliente"] = (
    "EC"
    + "|"
    + pan_visitas["compania__c"].astype(str)
    + "|"
    + pan_visitas["codigo_cliente__c"].astype(str)
)

In [30]:
# 1. Ordenar: los nulos primero, las fechas más recientes al final
pan_visitas = pan_visitas.sort_values(
    by=["codigo_unico__c", "ultima_visita"], 
    ascending=True, 
    na_position="first"
)

# 2. Eliminar duplicados manteniendo solo la última fila de cada cliente
pan_visitas = pan_visitas.drop_duplicates(subset="codigo_unico__c", keep="last")

In [31]:
pan_visitas.codigo_unico__c.nunique()

3904

In [32]:
# Combina ventas y visitas (nos quedamos con ruta y módulo de visitas si es nulo, usamos el valor de ventas)
df_merged = pd.merge(
    pan_ventas,
    pan_visitas[
        [
            "id_cliente",
            "dias_de_visita__c",
            "periodo_de_visita__c",
            "ultima_visita",
            "cod_ruta",
            "cod_modulo",
            "eje_potencial__c",
        ]
    ],
    on="id_cliente",
    # how='outer',  # Este es un merge 'outer', para conservar todos los clientes de ambos DataFrames.
    how="inner",
    suffixes=(
        "_df1",
        "_df2",
    ),  # Añadimos sufijos para diferenciar las columnas que se solapan
)

# Reemplazar los valores nulos de 'cod_ruta_df2' con los valores de 'cod_ruta_df1', y viceversa
# Si 'cod_ruta_df2' es nulo, se usará el valor de 'cod_ruta_df1'. Si ambos son nulos, se mantendrá nulo.
df_merged["cod_ruta"] = df_merged["cod_ruta_df2"].combine_first(
    df_merged["cod_ruta_df1"]
)
df_merged["cod_modulo"] = df_merged["cod_modulo_df2"].combine_first(
    df_merged["cod_modulo_df1"]
)

# Eliminar las columnas innecesarias ('cod_ruta_df1', 'cod_ruta_df2', 'cod_modulo_df1', 'cod_modulo_df2')
df_merged = df_merged.drop(
    columns=["cod_ruta_df1", "cod_ruta_df2", "cod_modulo_df1", "cod_modulo_df2"]
)

In [33]:
pan_ventas = df_merged.copy()
# Liberar memoria del df_merged
df_merged = None 
gc.collect()

25

In [34]:
pan_ventas["cod_cliente"] = pan_ventas["cod_cliente"].astype(str)
pan_ventas["cod_ruta"] = pan_ventas["cod_ruta"].astype(int)
pan_ventas["cod_modulo"] = pan_ventas["cod_modulo"].astype(int)

In [35]:
pan_ventas.cod_compania.nunique()

2

In [36]:
pan_ventas.cod_ruta.nunique()

16

In [37]:
pan_ventas[pan_ventas.cod_modulo.isin([81004, 81009, 81005, 81013])][["dias_de_visita__c","cod_modulo"]].drop_duplicates()

,dias_de_visita__c,cod_modulo
14,2;5,81005
36,2;5,81004
53,1;4,81009
79,3;6,81013


In [38]:
pan_ventas.groupby(["cod_compania","cod_modulo"]).agg(
    {"cod_articulo_magic": "nunique", "id_cliente": "nunique", "cod_ruta": "nunique"}
).sort_values("id_cliente")

cod_articulo_magic  id_cliente  cod_ruta
cod_compania cod_modulo                                          
6003         80332                        5           4         1
6010         81007                       91          57         1
6003         80328                       95          63         1
             80305                      101          65         1
6010         81011                       93          66         1
6003         80314                      102          67         1
6010         81004                      101          73         1
             81018                       95          73         1
             81005                      102          74         1
6003         80310                      102          74         1
6010         81021                      106          76         1
             81012                       89          78         1
6003         80329                      101          78         1
             80323                      102          79         1
             80327                       95          82         1
             80309                      100          83         1
             80302                       95          85         1
6010         81003                      102          85         1
6003         80317                      104          85         1
6010         81006                       97          86         1
             81015                       95          86         1
6003         80311                      108          87         1
             80306                       98          87         1
6010         81008                       95          87         1
6003         80320                      100          87         1
             80304                      104          87         1
             80307                      102          88         1
             80321                      102          88         1
             80301                       96          88         1
             80322                      103          89         1
6010         81009                       94          89         1
6003         80316                      106          90         1
             80308                       96          91         1
             80330                      102          91         1
             80324                       98          94         1
             80303                       97          94         1
             80312                      102          95         1
             80318                      103          96         1
6010         81014                       98          98         1
6003         80319                       96          99         1
             80315                      105         101         1
6010         81010                      111         101         1
             81016                      104         102         1
             81002                      101         103         1
             81013                      102         105         1
             81001                       94         118         1

# 2 Ajustar DF de clientes: Se filtrará a los clientes que tienen visita al día siguiente

## 2.2 Combinar Ventas con Subgiro, Descripcion de productos, Segmento del cliente y Fecha de creacion

In [39]:
# CAMBIAR DEPORADE POR SPORADE
pan_ventas["desc_marca"] = pan_ventas["desc_marca"].str.strip()
pan_ventas["desc_marca"] = pan_ventas["desc_marca"].replace({"DEPORADE": "SPORADE"})

pan_prod = pd.read_csv("Input/EC_maestro_productos.csv")

# Crear ID_CLIENTE
# PAN_VENTAS
pan_ventas["cod_sucursal"] = pan_ventas["cod_sucursal"].astype(str)

# PAN SEGMENTOS
pan_ventas = pan_ventas.rename(columns={"eje_potencial__c": "new_segment"})
# mapear valores
mapping = {
    "S1": "BLINDAR",
    "S2": "DESARROLLAR",
    "S4": "MANTENER",
    "S5": "OPTIMIZAR",
}

pan_ventas["new_segment"] = pan_ventas["new_segment"].map(mapping)
pan_ventas["new_segment"] = pan_ventas["new_segment"].fillna("OPTIMIZAR")

In [40]:
pan_ventas = pd.merge(
    pan_ventas,
    pan_prod[["cod_articulo_magic", "desc_articulo"]],
    how="left",
    on="cod_articulo_magic",
)

pan_ventas["mes"] = pd.to_datetime(pan_ventas["fecha_liquidacion"]).dt.strftime(
    "%Y-%m-01"
)

In [41]:
pan_ventas.desc_formato=pan_ventas.desc_formato.astype(str)

In [42]:
pan_ventas.to_parquet("Processed/ventas_ecuador_12m.parquet", index=False)

## 2.4 Filtrar clientes a visitar mañana

In [43]:
import random

In [44]:
# pan_ventas = pd.read_parquet("Processed/ventas_ecuador_12m.parquet")

In [45]:
pan_ventas.id_cliente.nunique()

3874

In [46]:
pan_ventas.groupby("dias_de_visita__c").id_cliente.nunique()

dias_de_visita__c
1;4       571
1;4;7     766
2;5      1189
3;6      1344
7           4
Name: id_cliente, dtype: int64

In [47]:
# Obtener el día de la semana actual (1 para lunes, 2 para martes, ..., 7 para domingo)
dia_actual = datetime.now(pytz.timezone("America/Lima")).weekday() + 1

# Si hoy es domingo (7), el día siguiente es lunes (1), de lo contrario, es el siguiente día al actual
if dia_actual == 6:
    dia_siguiente = 7
    # dia_siguiente=6
else:
    dia_siguiente = (dia_actual + 1) % 7
    # dia_siguiente = (dia_actual ) % 7

print("dia_actual", dia_actual)
print("dia_siguiente", dia_siguiente)

dia_actual 2
dia_siguiente 3


In [48]:
# Filtrar los clientes que serán visitados mañana
pan_ventas = pan_ventas[
    pan_ventas["dias_de_visita__c"]
    .astype(str)
    .apply(lambda x: str(dia_siguiente) in x.split(";"))
].reset_index(drop=True)

In [49]:
pan_ventas.groupby("cod_ruta")["dias_de_visita__c"].unique().reset_index()

,cod_ruta,dias_de_visita__c
0,80301,[3;6]
1,80302,[3;6]
2,80303,[3;6]
3,80304,[3;6]
4,80305,[3;6]
5,80306,[3;6]
6,80307,[3;6]
7,80308,[3;6]
8,80309,[3;6]
9,81001,[3;6]


In [50]:
pan_ventas.groupby("cod_ruta")["id_cliente"].nunique().describe()

count     15.000000
mean      89.600000
std        9.969955
min       66.000000
25%       84.500000
50%       91.000000
75%       95.500000
max      105.000000
Name: id_cliente, dtype: float64

In [51]:
pan_ventas.periodo_de_visita__c.unique()

array(['F1'], dtype=object)

In [52]:
pan_ventas.dias_de_visita__c.unique()

array(['3;6'], dtype=object)

In [53]:
pan_ventas.id_cliente.nunique()

1344

In [54]:
pan_ventas.to_parquet("Processed/ecuador_ventas_manana.parquet", index=False)

# 3. Pre Procesamiento

In [55]:
df_ventas = pd.read_parquet("Processed/ecuador_ventas_manana.parquet")

In [56]:
df_ventas["fecha_liquidacion"] = pd.to_datetime(
    df_ventas["fecha_liquidacion"], format="%Y-%m-%d"
)
df_ventas["desc_marca"] = df_ventas["desc_marca"].str.strip()
df_ventas["desc_categoria"] = df_ventas["desc_categoria"].str.strip()

## 3.1 Mapeo de Pesos por Giro
Los pesos representan el orden en el que se recomendara la marca para cada cliente dependiendo del Giro

In [57]:
# Mapeo de pesos para ordenar las recomendaciones finales segun su importancia por giro
mapeo_pesos = {}
mapeo_diccionario = {}

In [58]:
df_ventas.desc_marca.unique()

array(['CIELO', 'CIFRUT', 'BIG', 'ORO', 'SPORADE', 'PULP', 'VOLT',
       'D GUSSTO', 'DILYTE', 'DESS', 'VIDA', 'NUTRIDOG GOLD', 'BLING',
       'BIO ALOE'], dtype=object)

In [59]:
for giro_v in df_ventas["desc_subgiro"].unique():
    temp = df_ventas[(df_ventas["desc_subgiro"] == giro_v)]
    ranks = temp.groupby("desc_categoria")["cant_cajafisica_vta"].sum().reset_index()
    ranks.columns = ["index","desc_categoria"]
    ranks = ranks.sort_values(by="desc_categoria", ascending=False)
    if len(list(ranks["desc_categoria"])) <= 5:
        ranks["Ranking"] = range(1, len(ranks) + 1)
    elif len(list(ranks["desc_categoria"])) > 5:
        a = list(ranks["desc_categoria"])
        b = [1, 1, 2, 2]
        # Calculamos el multiplicador para el mapeo de pesos segun la varianza
        if np.std(a) / np.mean(a) <= 1.2:
            multiplicador = 4
        else:
            multiplicador = 2
        if len(a) > 5:
            for i in range(4, len(a)):
                if a[3] <= a[i] * multiplicador:
                    b.append(3)
                else:
                    b.append(3 + i)
        ranks["Ranking"] = b

    print("*" * 20)
    print("GIRO: ", giro_v)
    print("Categorias: ", list(ranks["index"]))
    print("Counts: ", list(ranks["desc_categoria"]))
    print(ranks.set_index("index")["Ranking"].to_dict())
    mapeo_diccionario[giro_v] = ranks.set_index("index")["Ranking"].to_dict()

********************
GIRO:  BODEGAS, TIENDAS
Categorias:  ['GASEOSAS', 'ISOTONICA', 'NECTAR', 'JUGOS LIGEROS', 'AGUA', 'ENERGIZANTE', 'DETERGENTE', 'CAFE', 'CONSERVA DE PESCADO', 'MASCOTA', 'ACEITE', 'SUERO', 'DESINFECTANTE', 'GELATINA', 'SUAVIZANTES', 'LAVAVAJILLAS', 'MODIFICADORES LACTEOS', 'CATEGORIA DEFAULT', 'SNACKS', 'ARROZ', 'ALOE VERA', 'PASTAS', 'CEREALES']
Counts:  [176326.75, 110652.0, 78375.66666666667, 53288.999999999935, 28655.0, 27689.5, 2829.66666666667, 2338.979166666666, 1597.624999999981, 992.75, 808.666666666667, 529.999999999917, 363.416666666621, 335.0, 312.0, 138.0, 92.0, 39.75, 24.650000000000002, 6.46, 5.0, 2.6, 0.066666666667]
{'GASEOSAS': 1, 'ISOTONICA': 1, 'NECTAR': 2, 'JUGOS LIGEROS': 2, 'AGUA': 3, 'ENERGIZANTE': 3, 'DETERGENTE': 9, 'CAFE': 10, 'CONSERVA DE PESCADO': 11, 'MASCOTA': 12, 'ACEITE': 13, 'SUERO': 14, 'DESINFECTANTE': 15, 'GELATINA': 16, 'SUAVIZANTES': 17, 'LAVAVAJILLAS': 18, 'MODIFICADORES LACTEOS': 19, 'CATEGORIA DEFAULT': 20, 'SNACKS': 21, 'AR

# 4. Modelo Pedido Sugerido

## 4.1. Importando librerias necesarias

In [60]:
import os

from pyspark.sql import SparkSession, SQLContext
from pyspark.sql.types import StructType, StringType, DoubleType, IntegerType
from pyspark.sql.functions import (
    when,
    col,
    regexp_replace,
    concat,
    countDistinct,
    lit,
    monotonically_increasing_id,
    hash,
)

from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.recommendation import ALS
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

In [61]:
!java -version

openjdk version "11.0.30" 2026-01-20 LTS
OpenJDK Runtime Environment Corretto-11.0.30.7.1 (build 11.0.30+7-LTS)
OpenJDK 64-Bit Server VM Corretto-11.0.30.7.1 (build 11.0.30+7-LTS, mixed mode)


In [62]:
import os
# os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

## 4.2 Construccion del algoritmo ALS

In [63]:
def create_spark_session():
    """Inicializa una única sesión de Spark optimizada para el job."""
    spark = (
        SparkSession.builder
        .appName("SageMaker-ALS-Recommender")
        .config("spark.hadoop.hadoop.security.authentication", "simple")
        .config("spark.hadoop.hadoop.security.authorization", "false")
        .config("spark.sql.shuffle.partitions", "200")
        .config("spark.sql.execution.arrow.pyspark.enabled", "true")
        .getOrCreate()
    )
    return spark

In [64]:
def als_training_job(spark, ruta):
    
    ventas = (
        spark.read.format("csv")
        .options(header=True, delimiter=",")
        .load(f"Processed/rutas/D_{ruta}_ventas.csv")
    )

    # --- PREPARACIÓN DE DATOS ---
    ventas = ventas.withColumn("cod_articulo_magic", col("cod_articulo_magic").cast(IntegerType()))
    sku_len = ventas.select("cod_articulo_magic").distinct().count()
    
    ventas = ventas.na.drop(subset=["fecha_liquidacion"])
    
    # Calcular frecuencia (Esta será nuestra métrica implícita directa)
    ventas = ventas.groupBy(
        ["id_cliente", "cod_articulo_magic", "cod_compania", "cod_cliente"]
    ).agg(countDistinct("fecha_liquidacion").alias("frecuencia"))
    
    ventas = ventas.withColumn("clienteId", concat(col("cod_compania"), lit("|"), col("cod_cliente")))

    # --- MAPEO SEGURO DE IDs ---
    # Usamos StringIndexer para crear un ID numérico secuencial e inequívoco
    indexer = StringIndexer(inputCol="clienteId", outputCol="clienteId_numeric", handleInvalid="skip")
    ventas = indexer.fit(ventas).transform(ventas)

    # Renombrar frecuencia a rating y limpiar nulos
    als_records = ventas.withColumn("rating", col("frecuencia").cast("float"))
    als_records = als_records.withColumn("clienteId_numeric", col("clienteId_numeric").cast("integer"))
    
    als_records = als_records.select("clienteId", "clienteId_numeric", "cod_articulo_magic", "rating")
    als_records = als_records.dropDuplicates(["clienteId", "cod_articulo_magic"])
    als_records = als_records.dropna(subset=["clienteId_numeric", "cod_articulo_magic", "rating"])    

    # Validación de seguridad para evitar fallos del algoritmo
    if als_records.count() < 10:
        print(f"Pocos registros en esta ruta, omitiendo ALS.")
        return pd.DataFrame()

    # --- ENTRENAMIENTO ALS ---
    als = ALS(
        rank=10,
        maxIter=5,
        implicitPrefs=True, # Aquí la frecuencia pura es interpretada como confianza
        ratingCol="rating",
        itemCol="cod_articulo_magic",
        userCol="clienteId_numeric",
        coldStartStrategy="drop"
    )
    model_als = als.fit(als_records)

    # --- GENERAR RECOMENDACIONES ---
    recs = model_als.recommendForAllUsers(sku_len)
    recs = recs.select("clienteId_numeric", "recommendations.cod_articulo_magic")
    
    # Recuperar el string original de clienteId
    recs = recs.join(
        als_records.select("clienteId", "clienteId_numeric").dropDuplicates(),
        on="clienteId_numeric",
        how="left",
    )

    # --- TRANSFORMACIÓN A PANDAS ---
    recs_to_parse = recs.select("clienteId", "cod_articulo_magic").toPandas()
    if recs_to_parse.empty:
        return pd.DataFrame()

    lista_rec = [f"r{i+1}" for i in range(sku_len)]
    new_cols = pd.DataFrame(
        recs_to_parse["cod_articulo_magic"].tolist(),
        index=recs_to_parse.index,
        columns=lista_rec,
    )
    recs_to_parse = pd.concat([recs_to_parse, new_cols], axis=1)
    client_recs = pd.melt(recs_to_parse, id_vars=["clienteId"], value_vars=lista_rec)

    client_recs["compania"] = client_recs["clienteId"].str.split("|").str[-2].apply(lambda x: str(x).rjust(4, "0"))
    client_recs["cliente"] = client_recs["clienteId"].str.split("|").str[1]
    
    # Concatenamos el ID completo. Recuerda eliminar la concatenación extra en el script de Reglas de Negocio.
    client_recs["id_cliente"] = "EC|" + client_recs["compania"] + "|" + client_recs["cliente"]
    client_recs["cod_articulo_magic"] = client_recs["value"]

    client_recs = client_recs[["id_cliente", "cod_articulo_magic"]].drop_duplicates().reset_index(drop=True)
    
    return client_recs

## 4.3 Corriendo PS

In [65]:
df_ventas.cod_compania.unique()

array(['6010', '6003'], dtype=object)

In [66]:
df_ventas.cod_ruta.unique()

array([81002, 81004, 81001, 81006, 80303, 80309, 80308, 80307, 80305,
       80306, 81005, 80304, 80301, 80302, 81003])

In [67]:
def runALS():
    print("Iniciando Modelado ALS (Modo Implícito Optimizado)...")
    spark = create_spark_session()
    
    # Ruta del archivo que deseas verificar y eliminar si existe
    ruta_archivo = "Output/D_rutas_rec.csv"
    # Verificar si el archivo existe
    if os.path.exists(ruta_archivo):
        # Eliminar el archivo si existe
        os.remove(ruta_archivo)
        print(f"El archivo en la ruta {ruta_archivo} ha sido eliminado.")
    else:
        print(f"El archivo en la ruta {ruta_archivo} no existe.")
        
    # CORRIENDO ALS a nivel ruta
    # Por temas con Pypsark tenemos que guardar el df temporal de la compañia en un CSV
    lista_recomendaciones = []
    for ruta in df_ventas.cod_ruta.unique():
        print("*" * 21)
        print(f"Procesando ALS para ruta: {ruta}...")
        
        temp = df_ventas[df_ventas.cod_ruta==ruta].reset_index(drop=True)
        temp.to_csv(f"Processed/rutas/D_{ruta}_ventas.csv", index=False)
        sku_len = temp["cod_articulo_magic"].nunique()
        
        print(f"SKUs disponibles en ruta {ruta}:", sku_len)
        df_rec = als_training_job(spark, ruta)
        
        if not df_rec.empty:
                lista_recomendaciones.append(df_rec)

    # Consolidar y guardar
    if lista_recomendaciones:
        print("Consolidando todas las recomendaciones...")
        df_final_recs = pd.concat(lista_recomendaciones, ignore_index=True)
        df_final_recs = df_final_recs.drop_duplicates()
        df_final_recs.to_csv("Output/D_rutas_rec.csv", index=False, sep=";")
        print(f"Recomendaciones guardadas exitosamente.")
    else:
        print("No se generaron recomendaciones.")

    spark.stop()

In [68]:
%%time
runALS()

Iniciando Modelado ALS (Modo Implícito Optimizado)...


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/03/31 21:28:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
El archivo en la ruta Output/D_rutas_rec.csv ha sido eliminado.
*********************
Procesando ALS para ruta: 81002...
SKUs disponibles en ruta 81002: 98
26/03/31 21:28:55 WARN InstanceBuilder$NativeBLAS: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/03/31 21:28:55 WARN InstanceBuilder$NativeBLAS: Failed to load implementation from:dev.ludovic.netlib.blas.ForeignLinkerBLAS
26/03/31 21:28:55 WARN InstanceBuilder$NativeLAPACK: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


*********************
Procesando ALS para ruta: 81004...
SKUs disponibles en ruta 81004: 89
*********************
Procesando ALS para ruta: 81001...
SKUs disponibles en ruta 81001: 95
*********************
Procesando ALS para ruta: 81006...
SKUs disponibles en ruta 81006: 104
*********************
Procesando ALS para ruta: 80303...
SKUs disponibles en ruta 80303: 100
*********************
Procesando ALS para ruta: 80309...
SKUs disponibles en ruta 80309: 100
*********************
Procesando ALS para ruta: 80308...
SKUs disponibles en ruta 80308: 102
*********************
Procesando ALS para ruta: 80307...
SKUs disponibles en ruta 80307: 95
*********************
Procesando ALS para ruta: 80305...
SKUs disponibles en ruta 80305: 102
*********************
Procesando ALS para ruta: 80306...
SKUs disponibles en ruta 80306: 98
*********************
Procesando ALS para ruta: 81005...
SKUs disponibles en ruta 81005: 102
*********************
Procesando ALS para ruta: 80304...
SKUs disponibles 

# 5. Ajustes al output del algoritmo ALS

## 5.-9 Quedarnos con skus que hayan tenido ventas en los ultimos 30 dias

In [69]:
pan_rec = pd.read_csv("Output/D_rutas_rec.csv", sep=";")
# pan_rec["id_cliente"] = "EC|" + pan_rec["id_cliente"]

In [70]:
pan_rec = pd.merge(pan_rec, pan_ventas[["id_cliente","cod_ruta"]].drop_duplicates(),on="id_cliente",how="left")

In [71]:
pan_rec

,id_cliente,cod_articulo_magic,cod_ruta
0,EC|6010|3253,523078,81002
1,EC|6010|3207,508462,81002
2,EC|6010|3208,621870,81002
3,EC|6010|1838,514445,81002
4,EC|6010|3102,514422,81002
...,...,...,...
132673,EC|6010|1380,800073,81003
132674,EC|6010|1988,524379,81003
132675,EC|6010|1106,623109,81003
132676,EC|6010|3178,524055,81003


In [72]:
pan_rec.id_cliente.nunique()

1344

In [73]:
%%time
df_ventas = pd.read_parquet("Processed/ecuador_ventas_manana.parquet")

CPU times: user 707 ms, sys: 268 ms, total: 974 ms
Wall time: 402 ms


In [74]:
df_ventas["fecha_liquidacion"] = pd.to_datetime(df_ventas["fecha_liquidacion"])

In [75]:
# Definir el umbral de tiempo (últimos 30 días desde hoy)
# bajamos de 30 a 14 dias 2025-07-25
hoy = pd.Timestamp.today()
fecha_limite = hoy - pd.Timedelta(days=14)
print(fecha_limite)
# Filtrar ventas en los últimos 30 días
ventas_filtradas = df_ventas[df_ventas["fecha_liquidacion"] >= fecha_limite]

2026-03-17 21:29:51.538394


In [76]:
# Obtener los productos vendidos en cada ruta
productos_por_comp = ventas_filtradas.groupby("cod_ruta")["cod_articulo_magic"].unique().reset_index()

In [77]:
# Unir con los productos vendidos en cada ruta
recomendaciones_validas = pan_rec.merge(productos_por_comp, on="cod_ruta", how="inner")

In [78]:
recomendaciones_validas.columns = ['id_cliente', 'cod_articulo_magic', 'cod_ruta','lista_sku']

In [79]:
recomendaciones_validas.shape

(132678, 4)

In [80]:
recomendaciones_validas  = recomendaciones_validas[recomendaciones_validas.apply(lambda row: row["cod_articulo_magic"] in row["lista_sku"], axis=1)].reset_index(drop=True)

In [81]:
recomendaciones_validas.shape

(74412, 4)

In [82]:
recomendaciones_validas.head()

,id_cliente,cod_articulo_magic,cod_ruta,lista_sku
0,EC|6010|3253,523078,81002,"[524188, 517260, 511920, 523078, 524187, 50858..."
1,EC|6010|3207,508462,81002,"[524188, 517260, 511920, 523078, 524187, 50858..."
2,EC|6010|3208,621870,81002,"[524188, 517260, 511920, 523078, 524187, 50858..."
3,EC|6010|3102,514422,81002,"[524188, 517260, 511920, 523078, 524187, 50858..."
4,EC|6010|596,500010,81002,"[524188, 517260, 511920, 523078, 524187, 50858..."


In [83]:
pan_rec.shape

(132678, 3)

In [84]:
pan_rec.head()

,id_cliente,cod_articulo_magic,cod_ruta
0,EC|6010|3253,523078,81002
1,EC|6010|3207,508462,81002
2,EC|6010|3208,621870,81002
3,EC|6010|1838,514445,81002
4,EC|6010|3102,514422,81002


In [85]:
pan_rec = recomendaciones_validas[["id_cliente","cod_articulo_magic"]].reset_index(drop=True)

In [86]:
pan_rec.shape

(74412, 2)

In [87]:
pan_rec.head()

,id_cliente,cod_articulo_magic
0,EC|6010|3253,523078
1,EC|6010|3207,508462
2,EC|6010|3208,621870
3,EC|6010|3102,514422
4,EC|6010|596,500010


In [88]:
pan_rec.to_csv("Output/D_rutas_rec.csv", sep=";", index=False)

## 5.-8 Recomendar de acuerdo a Subida, Bajada, Mantener

In [89]:
pan_rec = pd.read_csv("Output/D_rutas_rec.csv", sep=";")
#pan_rec["id_cliente"] = "EC|" + pan_rec["id_cliente"]

In [90]:
pan_rec.shape

(74412, 2)

In [91]:
pan_rec = pd.merge(pan_rec,pan_ventas[["id_cliente","cod_ruta"]].drop_duplicates(),on="id_cliente",how="left")

In [92]:
# df_ventas = pd.read_csv("Processed/ecuador_ventas_manana.csv")

In [93]:
def desplazar_nulos_fila(fila):
    return pd.Series(sorted(fila, key=pd.isna), index=fila.index)

In [94]:
# Definir una función para asignar los valores subida, mantener y bajar
def clasificar_valor(x):
    if x > 0:
        return "S"
    elif x == 0:
        return "M"
    else:
        return "B"

In [95]:
# df_ventas['fecha_liquidacion'] = pd.to_datetime(df_ventas['fecha_liquidacion'])

# Definir la fecha de referencia (hoy)
fecha_actual = datetime.now(pytz.timezone("America/Lima"))#.strftime('%Y-%m-%d')
fecha_30dias_atras = (fecha_actual - pd.Timedelta(days=30)).strftime('%Y-%m-%d')
fecha_60dias_atras = (fecha_actual - pd.Timedelta(days=60)).strftime('%Y-%m-%d')

print("fecha_actual",fecha_actual)
print("fecha_30dias_atras",fecha_30dias_atras)
print("fecha_60dias_atras",fecha_60dias_atras)

fecha_actual 2026-03-31 16:29:53.286846-05:00
fecha_30dias_atras 2026-03-01
fecha_60dias_atras 2026-01-30


In [96]:
# Filtrar los últimos 30 días
df_ultimos_30 = df_ventas[(df_ventas['fecha_liquidacion'] > fecha_30dias_atras) 
                          & (df_ventas['fecha_liquidacion'] <= fecha_actual.strftime('%Y-%m-%d'))]

# Filtrar los días 31 a 60
df_31_60 = df_ventas[(df_ventas['fecha_liquidacion'] > fecha_60dias_atras)
              & (df_ventas['fecha_liquidacion'] <= fecha_30dias_atras)]

# Agrupar por codigo_usuario, ruta, codigo_producto y sumar ventas
ventas_ultimos_30 = df_ultimos_30.groupby(['cod_ruta', 'cod_articulo_magic'])['imp_neto_vta_mn'].sum().reset_index()
ventas_ultimos_30["mes"] = "0_30"
ventas_31_60 = df_31_60.groupby(['cod_ruta', 'cod_articulo_magic'])['imp_neto_vta_mn'].sum().reset_index()
ventas_31_60["mes"] = "31_60"

df_grouped = pd.concat([ventas_ultimos_30,ventas_31_60],axis=0,ignore_index=True)
df_grouped = pd.pivot_table(
    df_grouped,
    values="imp_neto_vta_mn",
    index=["cod_ruta", "cod_articulo_magic"],
    columns=["mes"],
    aggfunc="sum",
).reset_index()

# Rellenando con 0 si es nulo
df_grouped["0_30"] = df_grouped["0_30"].fillna(0)
df_grouped["31_60"] = df_grouped["31_60"].fillna(0)

# Asignando variacion de productos por mes
df_grouped["v1_2"] = (
    (df_grouped[df_grouped.columns[2]] - df_grouped[df_grouped.columns[3]])
    / df_grouped[df_grouped.columns[3]]
    * 100
)
df_grouped["v1_2"].replace([float('inf'), float('-inf')], -1, inplace=True)
df_grouped["v1_2"] = df_grouped["v1_2"].fillna(-1)
df_grouped["vp"] = (df_grouped["v1_2"])

/tmp/ipykernel_30712/4182818935.py:34: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_grouped["v1_2"].replace([float('inf'), float('-inf')], -1, inplace=True)


In [97]:
# Aplicar la función a la columna 'valor' y guardar el resultado en una nueva columna 'flag'
df_grouped["flag"] = df_grouped["vp"].apply(lambda x: clasificar_valor(x))
# Definir el mapeo de valores
mapeo_flag = {"S": 0, "M": 1, "B": 2}
# Aplicar el mapeo a la columna 'clasificacion' y guardar el resultado en una nueva columna 'flag_rank'
df_grouped["flag_rank"] = df_grouped["flag"].map(mapeo_flag)

In [98]:
df_grouped

mes,cod_ruta,cod_articulo_magic,0_30,31_60,v1_2,vp,flag,flag_rank
0,80301,500010,365.2165,1347.8261,-72.903292,-72.903292,B,2
1,80301,500147,2474.9570,1014.1738,144.036772,144.036772,S,0
2,80301,500187,13.3044,6.6522,100.000000,100.000000,S,0
3,80301,501450,86.9564,0.0000,-1.000000,-1.000000,B,2
4,80301,508462,50.2171,40.1736,25.000249,25.000249,S,0
...,...,...,...,...,...,...,...,...
1050,81006,624010,60.8693,86.3040,-29.471056,-29.471056,B,2
1051,81006,699814,20.8800,18.0090,15.942029,15.942029,S,0
1052,81006,699815,10.4400,31.0590,-66.386555,-66.386555,B,2
1053,81006,800072,13.4000,0.0000,-1.000000,-1.000000,B,2


In [99]:
pan_rec.shape

(74412, 3)

In [100]:
pan_rec = pd.merge(pan_rec,df_grouped[["cod_ruta", "cod_articulo_magic","flag_rank"]],on=["cod_ruta", "cod_articulo_magic"],how="left")

In [101]:
pan_rec["flag_rank"] = pan_rec["flag_rank"].fillna(3)

In [102]:
# Agregar una columna con el orden original
pan_rec["original_order"] = pan_rec.index

In [103]:
# Ordenar por cod_cliente, rank_producto (ascendente) y mantener el orden original en caso de empate
pan_rec = pan_rec.sort_values(by=["id_cliente", "flag_rank", "original_order"], ascending=[True, True, True])#.drop(columns=["original_order"])

In [104]:
pan_rec.head()

,id_cliente,cod_articulo_magic,cod_ruta,flag_rank,original_order
38188,EC|6003|1398,599653,80305,0,38188
38279,EC|6003|1398,515517,80305,0,38279
38367,EC|6003|1398,517260,80305,0,38367
38460,EC|6003|1398,511920,80305,0,38460
38551,EC|6003|1398,509728,80305,0,38551


In [105]:
pan_rec.shape

(74412, 5)

In [106]:
pan_rec = pan_rec[["id_cliente","cod_articulo_magic"]].reset_index(drop=True)

In [107]:
pan_rec.head()

,id_cliente,cod_articulo_magic
0,EC|6003|1398,599653
1,EC|6003|1398,515517
2,EC|6003|1398,517260
3,EC|6003|1398,511920
4,EC|6003|1398,509728


In [108]:
pan_rec.shape

(74412, 2)

In [109]:
pan_rec.to_csv("Output/D_rutas_rec.csv", sep=";", index=False)

## 5.-5 Filtrar SKUs por STOCK

In [110]:
stock = pd.read_csv("s3://aje-prod-datalake-399723489351-us-east-2-landing-s3/analytics/pedido_sugerido/sellout/data/ecuador/D_stock_ec.csv")

/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/fsspec/registry.py:301: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)


In [111]:
stock

,COMPANIA,SUCURSAL,ARTICULO,STOCK,Fecha,Database
0,6002,1,500010,501,20260331,ECBDECO
1,6002,1,500147,621,20260331,ECBDECO
2,6002,1,500187,63,20260331,ECBDECO
3,6002,1,500302,0,20260331,ECBDECO
4,6002,1,500354,0,20260331,ECBDECO
...,...,...,...,...,...,...
4743,6001,1,699850,0,20260331,ECBDECO
4744,6001,1,699851,0,20260331,ECBDECO
4745,6001,1,800072,33,20260331,ECBDECO
4746,6001,1,800073,28,20260331,ECBDECO


In [112]:
stock.drop(columns=["Fecha","Database"], inplace=True)
stock.columns=["cod_compania","cod_sucursal","cod_articulo_magic","stock_cf"]

In [113]:
# stock["cod_compania"] = (stock["cod_compania"].astype(str).apply(lambda x: str(int(x)).rjust(4, "0")))
stock["cod_compania"] = stock["cod_compania"].astype(str).str.zfill(4)
stock["cod_sucursal"] = (stock["cod_sucursal"].astype(str).apply(lambda x: str(int(x)).rjust(2, "0")))

In [114]:
fecha_hace_12_dias = (datetime.today() - timedelta(days=12)).strftime('%Y-%m-%d')
print(fecha_hace_12_dias)

2026-03-19


In [115]:
# Promedio diario de cajas fisicas por sucursal y compañia de los ultimos 12 dias
prom_diario_vta = df_ventas[df_ventas.cant_cajafisica_vta > 0]
prom_diario_vta = df_ventas[df_ventas.fecha_liquidacion>=fecha_hace_12_dias].groupby(
    ["cod_compania","cod_sucursal","cod_articulo_magic","fecha_liquidacion"]).cant_cajafisica_vta.sum().reset_index().groupby(
    ["cod_compania","cod_sucursal","cod_articulo_magic"]).cant_cajafisica_vta.mean().reset_index()
prom_diario_vta["cod_compania"] = (prom_diario_vta["cod_compania"].astype(str).apply(lambda x: str(int(x)).rjust(4, "0")))
prom_diario_vta["cod_sucursal"] = (prom_diario_vta["cod_sucursal"].astype(str).apply(lambda x: str(int(x)).rjust(2, "0")))

In [116]:
df_stock = pd.merge(prom_diario_vta,stock, on=["cod_compania","cod_sucursal","cod_articulo_magic"],how="left")
df_stock["dias_stock"] = df_stock["stock_cf"]/df_stock["cant_cajafisica_vta"]

In [117]:
# temp_stock = df_stock.merge(df_ventas[["cod_articulo_magic","desc_marca","desc_articulo"]].drop_duplicates(), on="cod_articulo_magic", how="left")
# # Evalúa la condición y convierte True/False a 1/0
# condicion = ~((temp_stock["dias_stock"] > 3) & (temp_stock["cant_cajafisica_vta"] > 0))

# temp_stock["rechazo"] = condicion.astype(int)

# temp_stock.to_csv("Output/temp_stock.csv",index=False)

In [118]:
df_stock =  df_stock[(df_stock.dias_stock > 3) & (df_stock.cant_cajafisica_vta > 0)].reset_index(drop=True)

In [119]:
pan_rec.nunique()

id_cliente            1344
cod_articulo_magic      90
dtype: int64

In [120]:
pan_rec.groupby("id_cliente").cod_articulo_magic.nunique().mean()

55.36607142857143

In [121]:
pan_rec.shape

(74412, 2)

In [122]:
# Cruzamos stock

In [123]:
test = pd.merge(pan_rec,df_ventas[["id_cliente","cod_compania","cod_sucursal"]].drop_duplicates(),on="id_cliente",how="left")

In [124]:
test["cod_compania"] = (test["cod_compania"].astype(str).apply(lambda x: str(int(x)).rjust(4, "0")))
test["cod_sucursal"] = (test["cod_sucursal"].astype(str).apply(lambda x: str(int(x)).rjust(2, "0")))

In [125]:
test_f = pd.merge(test, df_stock, on=["cod_compania","cod_sucursal","cod_articulo_magic"], how="inner")

In [126]:
test_f.head()

,id_cliente,cod_articulo_magic,cod_compania,cod_sucursal,cant_cajafisica_vta,stock_cf,dias_stock
0,EC|6003|1398,515517,6003,01,87.50,1476,16.868571
1,EC|6003|1398,517260,6003,01,68.00,891,13.102941
2,EC|6003|1398,511920,6003,01,123.00,808,6.569106
3,EC|6003|1398,509728,6003,01,137.00,565,4.124088
4,EC|6003|1398,523578,6003,01,158.25,666,4.208531


In [127]:
test_f.shape

(56919, 7)

In [128]:
test_f.nunique()

id_cliente             1344
cod_articulo_magic       82
cod_compania              2
cod_sucursal              1
cant_cajafisica_vta      88
stock_cf                129
dias_stock              129
dtype: int64

In [129]:
PS_VAL_promedio_recs_por_cliente_stock = (
    test_f.groupby("id_cliente")["cod_articulo_magic"].nunique().mean()
)
PS_VAL_promedio_recs_por_cliente_stock

42.35044642857143

In [130]:
pan_rec = test_f[["id_cliente","cod_articulo_magic"]].copy()

In [131]:
pan_rec.to_csv("Output/D_rutas_rec.csv", sep=";", index=False)

## 5.-4 Mantener Solo SKUS en especifico (solo usar si se tiene el excel con SKU activos -> el input de ventas ya deberia incluir este filtro)

In [132]:
# pan_rec=pd.read_csv("Output/D_rutas_rec.csv", sep=";")
# pan_rec["id_cliente"] = "EC|" + pan_rec["id_cliente"]

In [133]:
# pan_rec.groupby("id_cliente")["cod_articulo_magic"].nunique().mean()

In [134]:
# pan_rec.groupby("id_cliente")["cod_articulo_magic"].count().mean()

In [135]:
# sku_con_precio=pd.read_excel("Input/LISTA DE PRECIOS ECUADOR_2024-11-08.xlsx",sheet_name="Hoja1")

In [136]:
# # Original size de recomendacioness
# pan_rec.shape

In [137]:
# pan_rec = pan_rec[pan_rec.cod_articulo_magic.isin(sku_con_precio["SKU"].unique())].reset_index(drop=True)

In [138]:
# # Size de recomendaciones luego de filtar solo a los del excel de SKUs activos
# pan_rec.shape

In [139]:
# pan_rec.groupby("id_cliente")["cod_articulo_magic"].count().mean()

In [140]:
# pan_rec["cod_articulo_magic"].nunique()

In [141]:
# pan_rec.to_csv("Output/D_rutas_rec.csv", sep=";",index=False)

## 5.-3 Quitar Recomendaciones de SKUS en especifico (NO PARA ECONOREDES AÚN)

In [142]:
pan_rec=pd.read_csv("Output/D_rutas_rec.csv", sep=";")
#pan_rec["id_cliente"]="EC|"+pan_rec["id_cliente"]

In [143]:
skus_sin_precio=[508585,516617,514445,514503,515907,516810,509727,511922,599890]

In [144]:
pan_rec.shape

(56919, 2)

In [145]:
pan_rec=pan_rec[~(pan_rec["cod_articulo_magic"].isin(skus_sin_precio))].reset_index(drop=True)

In [146]:
pan_rec.to_csv("Output/D_rutas_rec.csv", sep=";", index=False)

## 5.-2 Quitar Recomendaciones de los ultimos 14 dias

In [147]:
pan_rec=pd.read_csv("Output/D_rutas_rec.csv", sep=";")

In [148]:
master_prod=pd.read_csv("Input/EC_maestro_productos.csv")
master_prod

,cod_articulo_magic,desc_articulo
0,590908,AURRERA NARANJA PET NO RETORNABLE 3300 ML 6 PAC
1,590911,BIG CIFRUT CITRUS PET NO RETORNABLE 400 ML 12 ...
2,591102,AURRERA NARANJADA PET NO RETORNABLE 3000 ML 6 ...
3,591201,AURRERA PIÑA PET NO RETORNABLE 3000 ML 6 PACK
4,591307,AURRERA TORONJA PET NO RETORNABLE 3300 ML 6 PACK
...,...,...
7951,51359,TERMO MENUDERO MEDIANO BIG 70 X 86 CM
7952,624498,D GUSSTO CAFE LIOFILIZADO 50 GRS 24
7953,624532,DESS DESINFECTANTE LAVANDA 900 ML 6
7954,624531,DESS LAVAVAJILLAS LIMON 500 ML 12


In [149]:
fecha_tomorrow = (
    datetime.now(pytz.timezone("America/Lima")) + timedelta(days=1)
).strftime("%Y-%m-%d")
fecha_tomorrow

'2026-04-01'

In [150]:
# 2. La ruta (Key) del archivo en el bucket (sin el s3://...)
ruta_archivo = f"Econoredes/Ecuador/Output/PS_piloto_v1/D_base_pedidos_{fecha_tomorrow}.csv"

try:
    # Intentamos borrar el objeto directamente
    s3.delete_object(Bucket=bucket_name, Key=ruta_archivo)
    print(f"Intento de eliminación ejecutado para: s3://{bucket_name}/{ruta_archivo}")
except Exception as e:
    print(f"Ocurrió un error al intentar eliminar el archivo: {e}")
    print(f"Aún no existe: {ruta_archivo}")

Ocurrió un error al intentar eliminar el archivo: name 's3' is not defined
Aún no existe: Econoredes/Ecuador/Output/PS_piloto_v1/D_base_pedidos_2026-04-01.csv


In [151]:
# Configuración de S3
s3 = boto3.client('s3')
bucket_name = 'aje-analytics-ps-backup'
prefijo = 'Econoredes/Ecuador/Output/PS_piloto_v1/'

# Expresión regular para encontrar fechas en el nombre de los archivos
patron_fecha = r'\d{4}-\d{2}-\d{2}'

# Lista para almacenar las fechas extraídas
fechas = []

# Listar objetos dentro del bucket con el prefijo indicado
respuesta = s3.list_objects_v2(Bucket=bucket_name, Prefix=prefijo)

if 'Contents' in respuesta:
    for obj in respuesta['Contents']:
        # obj['Key'] contiene la ruta completa del archivo en S3
        ruta_completa = obj['Key']
        
        # Extraemos solo el nombre del archivo (ignorando las carpetas previas)
        archivo = ruta_completa.split('/')[-1]
        
        # Tu lógica original intacta
        if archivo.endswith('.csv') and re.match(r'^D_base_pedidos_\d{4}-\d{2}-\d{2}\.csv$', archivo):
            fecha = re.search(patron_fecha, archivo).group()
            fechas.append(fecha)

print("Fechas encontradas en S3:", fechas)

Fechas encontradas en S3: ['2026-03-23', '2026-03-24', '2026-03-25', '2026-03-26', '2026-03-27', '2026-03-28', '2026-03-30', '2026-03-31']


In [152]:
# 1. Definir hoy y el límite de hace 14 días (ambos como fechas)
hoy = datetime.today().date()
hace_14_dias = hoy - timedelta(days=14)

# 2. Filtrar la lista
last_7_days_2 = [
    f for f in fechas 
    if hace_14_dias <= datetime.strptime(f, "%Y-%m-%d").date() <= hoy
]

print("Fechas válidas:", last_7_days_2)

Fechas válidas: ['2026-03-23', '2026-03-24', '2026-03-25', '2026-03-26', '2026-03-27', '2026-03-28', '2026-03-30', '2026-03-31']


In [153]:
last_7_days_recs = pd.DataFrame()

for fecha_rec in last_7_days_2:
    # Construir la ruta completa del bucket S3
    ruta_s3 = f"s3://{bucket_name}/{prefijo}D_base_pedidos_{fecha_rec}.csv"
    
    try:
        # Pandas puede leer rutas s3:// directamente
        df_temp = pd.read_csv(ruta_s3, dtype={"Compania": "str", "Cliente": "str"})
        
        df_temp["id_cliente"] = 'EC|' + df_temp['Compania'] + '|' + df_temp['Cliente']
        df_temp = df_temp[df_temp["id_cliente"].isin(pan_rec["id_cliente"].unique())]
        
        last_7_days_recs = pd.concat([last_7_days_recs, df_temp], axis=0)
        print(f"{fecha_rec} done")
        
    except FileNotFoundError:
        print(f"El archivo para la fecha {fecha_rec} no se encontró en S3.")
    except Exception as e:
        print(f"Error al leer {fecha_rec} en S3: {e}")

2026-03-23 done
2026-03-24 done
2026-03-25 done
2026-03-26 done
2026-03-27 done
2026-03-28 done
2026-03-30 done
2026-03-31 done


In [154]:
last_7_days_recs

,Pais,Compania,Sucursal,Cliente,Modulo,Producto,Cajas,Unidades,Fecha,tipoRecomendacion,id_cliente,ultFecha,Destacar
0,EC,6003,1,1398,80312,599653,1,0,2026-03-25,PS1,EC|6003|1398,NaN,True
1,EC,6003,1,1398,80312,515417,1,0,2026-03-25,PS2,EC|6003|1398,NaN,True
2,EC,6003,1,1398,80312,509545,1,0,2026-03-25,PS3,EC|6003|1398,NaN,True
3,EC,6003,1,1399,80312,599653,1,0,2026-03-25,PS1,EC|6003|1399,NaN,True
4,EC,6003,1,1399,80312,516617,1,0,2026-03-25,PS2,EC|6003|1399,NaN,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3468,EC,6010,1,997,81016,515519,1,0,2026-03-28,PS2,EC|6010|997,NaN,True
3469,EC,6010,1,999,81013,523108,1,0,2026-03-28,PS1,EC|6010|999,NaN,True
3470,EC,6010,1,999,81013,515519,1,0,2026-03-28,PS2,EC|6010|999,NaN,True
3471,EC,6010,1,999,81013,522514,1,0,2026-03-28,PS3,EC|6010|999,NaN,True


In [155]:
# Utilizamos merge para combinar los dataframes, utilizando la columna 'Cliente' como clave
df_combinado = pd.merge(pan_rec, last_7_days_recs, left_on=['id_cliente', 'cod_articulo_magic'],right_on=['id_cliente', 'Producto'], how='left', indicator=True)
# Filtramos los registros que solo están en el DataFrame 1
df_resultado = df_combinado[df_combinado['_merge'] == 'left_only'][["id_cliente","cod_articulo_magic"]].reset_index(drop=True)
df_resultado

,id_cliente,cod_articulo_magic
0,EC|6003|1398,517260
1,EC|6003|1398,511920
2,EC|6003|1398,509728
3,EC|6003|1398,523578
4,EC|6003|1398,515904
...,...,...
50229,EC|6010|999,524382
50230,EC|6010|999,622131
50231,EC|6010|999,623821
50232,EC|6010|999,624010


In [156]:
df_resultado.shape

(50234, 2)

In [157]:
df_resultado.to_csv("Output/D_rutas_rec.csv", sep=";", index=False)

## 5.0 Calcular Irregularidad de clientes

In [158]:
pan_rec = pd.read_csv("Output/D_rutas_rec.csv", sep=";")

In [159]:
pan_rec["id_cliente"].nunique()

1344

In [160]:
now = pd.to_datetime(datetime.now(pytz.timezone("America/Lima")).strftime("%Y-%m-01"))
fecha_doce_meses_atras = now - pd.DateOffset(months=12)
lista_m12 = [fecha_doce_meses_atras + pd.DateOffset(months=i) for i in range(12)]
lista_m12

fecha_seis_meses_atras = now - pd.DateOffset(months=6)
lista_m6 = [fecha_seis_meses_atras + pd.DateOffset(months=i) for i in range(6)]
lista_m6

[Timestamp('2025-09-01 00:00:00'),
 Timestamp('2025-10-01 00:00:00'),
 Timestamp('2025-11-01 00:00:00'),
 Timestamp('2025-12-01 00:00:00'),
 Timestamp('2026-01-01 00:00:00'),
 Timestamp('2026-02-01 00:00:00')]

In [161]:
temp_df_ventas = df_ventas[df_ventas["id_cliente"].isin(pan_rec["id_cliente"].unique())]

In [162]:
qw = (
    temp_df_ventas[["id_cliente", "mes"]]
    .drop_duplicates()
    .sort_values(["id_cliente", "mes"])
    .groupby("id_cliente")
    .tail(12)
    .reset_index(drop=True)
)
qw["mes"] = pd.to_datetime(qw["mes"])
qw["m12"] = qw["mes"].isin(lista_m12)
qw["m6"] = qw["mes"].isin(lista_m6)

In [163]:
categoria_cliente = qw.groupby("id_cliente")[["m12", "m6"]].sum().reset_index()

In [164]:
# Definir condiciones
condicion_super = categoria_cliente["m12"] == 12
condicion_frecuente = (
    (categoria_cliente["m12"] < 12)
    & (categoria_cliente["m12"] >= 6)
    & (categoria_cliente["m6"] == 6)
)
condicion_regulares = (categoria_cliente["m6"] <= 6) & (categoria_cliente["m12"] <= 6)
condicion_riesgo = (categoria_cliente["m6"] < 6) & (categoria_cliente["m6"] >= 4)
condicion_irregulares = categoria_cliente["m6"] < 4

# Aplicar condiciones y asignar categorías
categoria_cliente.loc[condicion_super, "categoria_cliente"] = "Super"
categoria_cliente.loc[condicion_frecuente, "categoria_cliente"] = "Frecuente"
categoria_cliente.loc[condicion_regulares, "categoria_cliente"] = "Regular"
categoria_cliente.loc[condicion_riesgo, "categoria_cliente"] = "Riesgo"
categoria_cliente.loc[condicion_irregulares, "categoria_cliente"] = "Irregular"

**Recalcular Irregularidad**

In [165]:
# Definir condiciones
condicion_regular = (
    (categoria_cliente["categoria_cliente"] == "Super")
    | (categoria_cliente["categoria_cliente"] == "Frecuente")
    | (categoria_cliente["categoria_cliente"] == "Regular")
    | (categoria_cliente["categoria_cliente"] == "Riesgo")
)
condicion_irregular = categoria_cliente["categoria_cliente"] == "Irregular"
# Aplicar condiciones y asignar categorías
categoria_cliente.loc[condicion_regular, "categoria_cliente_2"] = "Regular"
categoria_cliente.loc[condicion_irregular, "categoria_cliente_2"] = "Irregular"

In [166]:
categoria_cliente

,id_cliente,m12,m6,categoria_cliente,categoria_cliente_2
0,EC|6003|1398,11,6,Frecuente,Regular
1,EC|6003|1399,11,6,Frecuente,Regular
2,EC|6003|1400,11,6,Frecuente,Regular
3,EC|6003|1403,11,6,Frecuente,Regular
4,EC|6003|1405,11,6,Frecuente,Regular
...,...,...,...,...,...
1339,EC|6010|992,11,6,Frecuente,Regular
1340,EC|6010|993,11,6,Frecuente,Regular
1341,EC|6010|996,11,6,Frecuente,Regular
1342,EC|6010|997,11,6,Frecuente,Regular


In [167]:
categoria_cliente["categoria_cliente"].value_counts(dropna=False)

categoria_cliente
Frecuente    1062
Riesgo        141
Irregular     109
Super          23
Regular         9
Name: count, dtype: int64

In [168]:
categoria_cliente["categoria_cliente_2"].value_counts(dropna=False)

categoria_cliente_2
Regular      1235
Irregular     109
Name: count, dtype: int64

## 5.1. Obteniendo marcas de recomendaciones por cliente

In [169]:
pan_rec = pd.read_csv("Output/D_rutas_rec.csv", sep=";")

In [170]:
marca_articulo = df_ventas[["desc_categoria", "cod_articulo_magic"]].drop_duplicates()
print(marca_articulo.shape)
marca_articulo.head()

(130, 2)


,desc_categoria,cod_articulo_magic
0,AGUA,508462
1,JUGOS LIGEROS,509728
2,GASEOSAS,599653
3,GASEOSAS,523100
4,ISOTONICA,515417


In [171]:
cliente_rec_marca = pd.merge(
    pan_rec, marca_articulo, on="cod_articulo_magic", how="left"
)
cliente_rec_marca["desc_categoria"] = cliente_rec_marca["desc_categoria"].str.strip()
print(cliente_rec_marca.shape)
cliente_rec_marca

(50234, 3)


,id_cliente,cod_articulo_magic,desc_categoria
0,EC|6003|1398,517260,ENERGIZANTE
1,EC|6003|1398,511920,JUGOS LIGEROS
2,EC|6003|1398,509728,JUGOS LIGEROS
3,EC|6003|1398,523578,JUGOS LIGEROS
4,EC|6003|1398,515904,ISOTONICA
...,...,...,...
50229,EC|6010|999,524382,AGUA
50230,EC|6010|999,622131,MODIFICADORES LACTEOS
50231,EC|6010|999,623821,CONSERVA DE PESCADO
50232,EC|6010|999,624010,CAFE


## 5.2. Calcular marcas distintas recomendadas por cliente

In [172]:
cods2 = cliente_rec_marca.groupby("id_cliente")["desc_categoria"].nunique().reset_index()
print(cods2.shape)
cods2.head()

(1344, 2)


,id_cliente,desc_categoria
0,EC|6003|1398,14
1,EC|6003|1399,14
2,EC|6003|1400,14
3,EC|6003|1403,14
4,EC|6003|1405,14


## 5.3 Quitando los SKUs de las ultimas 2 semanas (Evitar recompra)

In [173]:
last_2_weeks = pd.to_datetime(datetime.now(pytz.timezone("America/Lima"))) - pd.DateOffset(
    days=14
)
# Quita la zona horaria para que coincida con el tipo datetime64[ns]
last_2_weeks_naive = last_2_weeks.tz_localize(None)
last_2_weeks_naive

Timestamp('2026-03-17 16:29:56.248990')

In [174]:
df_quitar_2_weeks = df_ventas[df_ventas["fecha_liquidacion"]>=last_2_weeks_naive][["id_cliente","cod_articulo_magic"]].drop_duplicates().reset_index(drop=True)

In [175]:
# df_quitar_2_weeks.to_csv("Output/temp_skus_quitados_por_cliente.csv",index=False)

In [176]:
# Hacemos un merge para identificar coincidencias
cliente_rec_sin4 = cliente_rec_marca.merge(df_quitar_2_weeks, on=['id_cliente', 'cod_articulo_magic'], how='left', indicator=True)
# Filtramos solo los registros que NO están en df_quitar_2_weeks
cliente_rec_sin4 = cliente_rec_sin4[cliente_rec_sin4['_merge'] == 'left_only'].drop(columns=['_merge'])

In [177]:
cliente_rec_marca.id_cliente.nunique()

1344

In [178]:
cliente_rec_sin4.id_cliente.nunique()

1344

In [179]:
cliente_rec_sin4

,id_cliente,cod_articulo_magic,desc_categoria
0,EC|6003|1398,517260,ENERGIZANTE
2,EC|6003|1398,509728,JUGOS LIGEROS
7,EC|6003|1398,599211,CONSERVA DE PESCADO
8,EC|6003|1398,514422,NECTAR
10,EC|6003|1398,621870,CAFE
...,...,...,...
50229,EC|6010|999,524382,AGUA
50230,EC|6010|999,622131,MODIFICADORES LACTEOS
50231,EC|6010|999,623821,CONSERVA DE PESCADO
50232,EC|6010|999,624010,CAFE


## 5.4 Filtro para calcula antiguedad de clientes

In [180]:
df_ventas = df_ventas.rename(columns={"fecha_creacion_cliente": "fecha_creacion"})

In [181]:
df_ventas[["fecha_creacion"]].head()

,fecha_creacion
0,2024-08-28 00:00:00
1,2020-11-25 00:00:00
2,2020-11-25 00:00:00
3,2020-11-25 00:00:00
4,2020-10-20 00:00:00


In [182]:
# Convertir la columna 'fecha_creacion' a tipo fecha
df_ventas["fecha_creacion"] = pd.to_datetime(
    df_ventas["fecha_creacion"], format="%Y-%m-%d %H:%M:%S"
)

In [183]:
# Definir una función para etiquetar los clientes según su fecha de creación
def etiquetar_cliente(fecha_creacion):
    if pd.isnull(fecha_creacion):
        return (
            "nf"  # Si la fecha de creación es nula, etiquetar como 'nf' (no encontrada)
        )
    else:
        hoy = datetime.now()  # Obtener la fecha actual
        hace_12_meses = hoy - timedelta(days=365)  # Calcular hace 12 meses
        if fecha_creacion >= hace_12_meses:
            return "new_client"  # Si la fecha de creación está dentro de los últimos 12 meses, etiquetar como 'new_client'
        else:
            return "old_client"  # Si la fecha de creación es anterior a los últimos 12 meses, etiquetar como 'old_client'

In [184]:
# Aplicar la función a la columna 'fecha_creacion' para crear la nueva columna 'tipo_cliente'
df_ventas["antiguedad"] = df_ventas["fecha_creacion"].apply(etiquetar_cliente)

In [185]:
pan_antiguedad_clientes = (
    df_ventas[["id_cliente", "fecha_creacion", "antiguedad"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

## 5.4.2 Obteniendo datos necesarios para armar DF para subir a SalesForce

In [186]:
datos_para_salesforce = (
    df_ventas.groupby(
        ["id_cliente", "cod_compania", "cod_sucursal", "cod_cliente", "cod_ruta"]
    )[
        [
            "id_cliente",
            "cod_compania",
            "cod_sucursal",
            "cod_cliente",
            "cod_modulo",
            "cod_ruta",
        ]
    ]
    .head(1)
    .reset_index(drop=True)
)

In [187]:
datos_para_salesforce["cod_cliente"]=datos_para_salesforce["cod_cliente"].astype(int)
datos_para_salesforce["cod_sucursal"]=datos_para_salesforce["cod_sucursal"].astype(int)
datos_para_salesforce["cod_compania"]=datos_para_salesforce["cod_compania"].astype(str)
datos_para_salesforce["cod_modulo"]=datos_para_salesforce["cod_modulo"].astype(int)
datos_para_salesforce["cod_ruta"]=datos_para_salesforce["cod_ruta"].astype(int)

## 5.5 Juntando las Recomendaciones Finales
 - Ordenado de acuerdo al peso de la recomendación y el id de usuario.
 - Hay 5 skus de 5 marcas distintas que se recomiendan a cada usuario
 - Ningún SKU que haya comprado el usuario en sus ultimas X visitas será recomendado. La cantidad de visitas (X) a ignorar depende de sus visitas promedio mensuales.

In [188]:
# # luego de quitar skus comprados en las ultimas 2 semanas
# cliente_rec_sin4.to_csv("Output/D_rutas_rec_5.csv",index=False)

In [189]:
# Obtener primer SKU de cada marca de la recomendacion
final_rec = cliente_rec_sin4.groupby(["id_cliente", "desc_categoria"]).first().reset_index()
# Agregar Marca de sus primeras compras
final_rec = pd.merge(final_rec, cods2, how="left", on="id_cliente")
# Agregar descripcion de SKU recomendado
pan_prod = pan_prod[["cod_articulo_magic", "desc_articulo"]].drop_duplicates()
pan_prod = pan_prod.groupby(["cod_articulo_magic"]).first().reset_index()
final_rec = pd.merge(final_rec, pan_prod, how="left", on="cod_articulo_magic")
# Cambiar nombre de columnas
final_rec.columns = ["id_cliente", "marca_rec", "sku", "len_marca_rec", "desc_rec"]
# Agregar Giro del cliente
giros = df_ventas[df_ventas["id_cliente"].isin(final_rec["id_cliente"].unique())][
    ["id_cliente", "desc_giro", "desc_subgiro"]
].drop_duplicates()
final_rec = pd.merge(final_rec, giros, on="id_cliente", how="left")
# Agregar si el cliente es Regular o Irregular
final_rec = pd.merge(
    final_rec,
    categoria_cliente[["id_cliente", "categoria_cliente_2"]],
    on="id_cliente",
    how="left",
)
final_rec.loc[final_rec["categoria_cliente_2"] == "Irregular", "desc_giro"] = np.nan
# Agregar antiguedad de cliente
final_rec = pd.merge(final_rec, pan_antiguedad_clientes, how="left", on="id_cliente")
# Agregar segmento de cliente
info_segmentos = pan_ventas[
    [
        "id_cliente",
        "new_segment",
        "dias_de_visita__c",
        "periodo_de_visita__c",
        "ultima_visita",
    ]
].drop_duplicates()
info_segmentos = info_segmentos.groupby(["id_cliente"]).first().reset_index()
final_rec = pd.merge(final_rec, info_segmentos, how="left", on="id_cliente")
# Agregar datos necesarios para SALESFORCE
final_rec = pd.merge(final_rec, datos_para_salesforce, how="left", on="id_cliente")
# Agregar Peso de marcas dependiendo del giro del cliente
final_rec["peso"] = final_rec.apply(
    lambda row: mapeo_diccionario.get(row["desc_subgiro"], {}).get(row["marca_rec"], 5),
    axis=1,
)

In [190]:
# Ordenar la recomendacion de acuerdo al peso de la marca 
final_rec = (
    final_rec.sort_values(["id_cliente", "peso"])
)
# Agregamos rank de la marca recomendada
final_rec["marca_rec_rank"] = final_rec.groupby("id_cliente").cumcount() + 1
final_rec = final_rec.reset_index(drop=True)

In [191]:
final_rec["sku"] = final_rec["sku"].astype(int)
final_rec["peso"] = final_rec["peso"].astype(int)
final_rec["marca_rec_rank"] = final_rec["marca_rec_rank"].astype(int)

In [192]:
final_rec.shape

(15735, 21)

In [193]:
final_rec.head()

,id_cliente,marca_rec,sku,len_marca_rec,desc_rec,desc_giro,desc_subgiro,categoria_cliente_2,fecha_creacion,antiguedad,...,dias_de_visita__c,periodo_de_visita__c,ultima_visita,cod_compania,cod_sucursal,cod_cliente,cod_modulo,cod_ruta,peso,marca_rec_rank
0,EC|6003|1398,GASEOSAS,523075,14,BIG MANZANA SIN AZUCAR PET NO RETORNABLE 1100 ...,BODEGA Y PARTICULARES,"BODEGAS, TIENDAS",Regular,2020-10-20,old_client,...,3;6,F1,2026-03-28,6003,1,1398,80312,80305,1,1
1,EC|6003|1398,JUGOS LIGEROS,509728,14,CIFRUT TROPICAL PUNCH PET NO RETORNABLE 1000 M...,BODEGA Y PARTICULARES,"BODEGAS, TIENDAS",Regular,2020-10-20,old_client,...,3;6,F1,2026-03-28,6003,1,1398,80312,80305,2,2
2,EC|6003|1398,NECTAR,514422,14,PULP DURAZNO TETRA PACK 1000 ml 6 PACK,BODEGA Y PARTICULARES,"BODEGAS, TIENDAS",Regular,2020-10-20,old_client,...,3;6,F1,2026-03-28,6003,1,1398,80312,80305,2,3
3,EC|6003|1398,AGUA,508514,14,CIELO AGUA SIN GAS PET NO RETORNABLE 3500 ml 6...,BODEGA Y PARTICULARES,"BODEGAS, TIENDAS",Regular,2020-10-20,old_client,...,3;6,F1,2026-03-28,6003,1,1398,80312,80305,3,4
4,EC|6003|1398,ENERGIZANTE,517260,14,VOLT BLUEBERRY PET NO RETORNABLE 400 ML 12,BODEGA Y PARTICULARES,"BODEGAS, TIENDAS",Regular,2020-10-20,old_client,...,3;6,F1,2026-03-28,6003,1,1398,80312,80305,3,5


In [194]:
# # luego de quedarnos con el 1ro por cateogria
# final_rec.to_csv("Output/D_rutas_rec_6.csv",index=False)

## 5.7 Filtro de Recomendaciones por Segmento de cliente

In [195]:
# Definir una función para filtrar las filas según el valor de 'new_segment'
def filtrar_segmento(group):
    segmento = group["new_segment"].iloc[
        0
    ]  # Obtener el valor del segmento para el grupo
    if segmento == "BLINDAR":
        return group.head(1)  # Si el segmento es 'Blindar', mantener la primera fila
    elif segmento == "MANTENER":
        return group.head(
            2
        )  # Si el segmento es 'Mantener', mantener las dos primeras filas
    elif segmento == "DESARROLLAR":
        return group.head(
            3
        )  # Si el segmento es 'Desarrollar', mantener las tres primeras filas
    elif segmento == "OPTIMIZAR":
        return group.head(
            4
        )  # Si el segmento es 'Optimizar', mantener las cuatro primeras filas
    else:
        return group  # Si el segmento no es ninguno de los especificados, mantener todas las filas

In [196]:
final_rec = (
    final_rec.groupby("id_cliente").apply(filtrar_segmento).reset_index(drop=True)
)

/tmp/ipykernel_30712/3325175710.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  final_rec.groupby("id_cliente").apply(filtrar_segmento).reset_index(drop=True)


In [197]:
final_rec.groupby("new_segment")["id_cliente"].nunique()

new_segment
BLINDAR        270
DESARROLLAR    295
MANTENER       399
OPTIMIZAR      380
Name: id_cliente, dtype: int64

## 5.8. Guardando recomendacion para D&A y Comercial

In [198]:
final_rec.id_cliente.nunique()

1344

In [199]:
final_rec.dias_de_visita__c.value_counts(dropna=False)

dias_de_visita__c
3;6    3473
Name: count, dtype: int64

In [200]:
final_rec.periodo_de_visita__c.value_counts(dropna=False)

periodo_de_visita__c
F1    3473
Name: count, dtype: int64

### 5.8.1 Para D&A

In [201]:
fecha_tomorrow = (
    datetime.now(pytz.timezone("America/Lima")) + timedelta(days=1)
).strftime("%Y-%m-%d")
fecha_tomorrow

'2026-04-01'

In [202]:
final_rec.shape

(3473, 21)

In [203]:
df_ultima_venta = df_ventas[["id_cliente","cod_articulo_magic","fecha_liquidacion"]].sort_values("fecha_liquidacion").drop_duplicates(
    subset=["id_cliente", "cod_articulo_magic"], 
    keep="last"
).reset_index(drop=True)
df_ultima_venta.rename(columns={"fecha_liquidacion":"ult_fecha_liquidacion"}, inplace=True)

In [204]:
extra_fr = final_rec.merge(df_ultima_venta, left_on=["id_cliente","sku"], right_on = ["id_cliente","cod_articulo_magic"], how="left")
extra_fr.drop(columns=["cod_articulo_magic"],inplace=True)

In [205]:
extra_fr.shape

(3473, 22)

In [206]:
extra_fr.head()

,id_cliente,marca_rec,sku,len_marca_rec,desc_rec,desc_giro,desc_subgiro,categoria_cliente_2,fecha_creacion,antiguedad,...,periodo_de_visita__c,ultima_visita,cod_compania,cod_sucursal,cod_cliente,cod_modulo,cod_ruta,peso,marca_rec_rank,ult_fecha_liquidacion
0,EC|6003|1398,GASEOSAS,523075,14,BIG MANZANA SIN AZUCAR PET NO RETORNABLE 1100 ...,BODEGA Y PARTICULARES,"BODEGAS, TIENDAS",Regular,2020-10-20,old_client,...,F1,2026-03-28,6003,1,1398,80312,80305,1,1,NaT
1,EC|6003|1398,JUGOS LIGEROS,509728,14,CIFRUT TROPICAL PUNCH PET NO RETORNABLE 1000 M...,BODEGA Y PARTICULARES,"BODEGAS, TIENDAS",Regular,2020-10-20,old_client,...,F1,2026-03-28,6003,1,1398,80312,80305,2,2,2026-02-15
2,EC|6003|1398,NECTAR,514422,14,PULP DURAZNO TETRA PACK 1000 ml 6 PACK,BODEGA Y PARTICULARES,"BODEGAS, TIENDAS",Regular,2020-10-20,old_client,...,F1,2026-03-28,6003,1,1398,80312,80305,2,3,2026-03-09
3,EC|6003|1399,GASEOSAS,523075,14,BIG MANZANA SIN AZUCAR PET NO RETORNABLE 1100 ...,BODEGA Y PARTICULARES,"BODEGAS, TIENDAS",Regular,2020-10-20,old_client,...,F1,2026-03-28,6003,1,1399,80312,80305,1,1,NaT
4,EC|6003|1399,ISOTONICA,515417,14,SPORADE UVA PET NO RETORNABLE 1200 ML 6,BODEGA Y PARTICULARES,"BODEGAS, TIENDAS",Regular,2020-10-20,old_client,...,F1,2026-03-28,6003,1,1399,80312,80305,1,2,2026-03-12


In [207]:
# # luego de filtro por segmento
# extra_fr.to_csv("Output/D_rutas_rec_7.csv",index=False)

In [208]:
# df_ventas[(df_ventas.id_cliente=="EC|6003|1050")&(df_ventas.cod_articulo_magic==523075)].fecha_liquidacion

In [209]:
extra_fr.to_csv(
    f"Output/PS_piloto_data_v1/D_recs_extra_data_{fecha_tomorrow}.csv", index=False
)

In [210]:
# Establecer la conexión con S3
bucket_name = 'aje-analytics-ps-backup'  # nombre de bucket en S3
file_name = f'Econoredes/Ecuador/Output/PS_piloto_data_v1/D_recs_extra_data_{fecha_tomorrow}.csv'  # nombre para el archivo en S3
s3_path = f's3://{bucket_name}/{file_name}'

# Escribir el dataframe en S3 con AWS Data Wrangler
wr.s3.to_csv(final_rec, s3_path, index=False)

{'paths': ['s3://aje-analytics-ps-backup/Econoredes/Ecuador/Output/PS_piloto_data_v1/D_recs_extra_data_2026-04-01.csv'],
 'partitions_values': {}}

### 5.8.2 Para subir a PS

In [211]:
recomendaciones_para_salesforce = final_rec[
    ["cod_compania", "cod_sucursal", "cod_cliente", "cod_modulo", "sku"]
]
recomendaciones_para_salesforce["Pais"] = "EC"
recomendaciones_para_salesforce["Cajas"] = int(1)
recomendaciones_para_salesforce["Unidades"] = int(0)
recomendaciones_para_salesforce["Fecha"] = fecha_tomorrow
recomendaciones_para_salesforce = recomendaciones_para_salesforce[
    [
        "Pais",
        "cod_compania",
        "cod_sucursal",
        "cod_cliente",
        "cod_modulo",
        "sku",
        "Cajas",
        "Unidades",
        "Fecha",
    ]
]
recomendaciones_para_salesforce.columns = [
    "Pais",
    "Compania",
    "Sucursal",
    "Cliente",
    "Modulo",
    "Producto",
    "Cajas",
    "Unidades",
    "Fecha",
]

/tmp/ipykernel_30712/4041848949.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  recomendaciones_para_salesforce["Pais"] = "EC"
/tmp/ipykernel_30712/4041848949.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  recomendaciones_para_salesforce["Cajas"] = int(1)
/tmp/ipykernel_30712/4041848949.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.

In [212]:
recomendaciones_para_salesforce["Compania"] = recomendaciones_para_salesforce["Compania"].apply(formatear_codigo)
recomendaciones_para_salesforce["Sucursal"] = recomendaciones_para_salesforce[
    "Sucursal"
].apply(lambda x: str(int(x)).rjust(2, "0"))

In [213]:
recomendaciones_para_salesforce.Sucursal.unique()

array(['01'], dtype=object)

In [214]:
recomendaciones_para_salesforce.Compania.unique()

array(['6003', '6010'], dtype=object)

In [215]:
recomendaciones_para_salesforce[["Compania","Cliente"]].drop_duplicates().shape

(1344, 2)

In [216]:
recomendaciones_para_salesforce

,Pais,Compania,Sucursal,Cliente,Modulo,Producto,Cajas,Unidades,Fecha
0,EC,6003,01,1398,80312,523075,1,0,2026-04-01
1,EC,6003,01,1398,80312,509728,1,0,2026-04-01
2,EC,6003,01,1398,80312,514422,1,0,2026-04-01
3,EC,6003,01,1399,80312,523075,1,0,2026-04-01
4,EC,6003,01,1399,80312,515417,1,0,2026-04-01
...,...,...,...,...,...,...,...,...,...
3468,EC,6010,01,997,81016,515517,1,0,2026-04-01
3469,EC,6010,01,999,81013,523104,1,0,2026-04-01
3470,EC,6010,01,999,81013,515904,1,0,2026-04-01
3471,EC,6010,01,999,81013,509728,1,0,2026-04-01


**Conteo de marcas recomendadas**

In [217]:
count = (
    final_rec.groupby("id_cliente")["marca_rec"]
    .apply(lambda x: "-".join((x)))
    .reset_index(name="marcas_ordenadas")
)
count = (
    count.groupby("marcas_ordenadas")["id_cliente"]
    .nunique()
    .sort_values(ascending=False)
)
count.head()

marcas_ordenadas
GASEOSAS-ISOTONICA                         342
GASEOSAS-ISOTONICA-JUGOS LIGEROS-NECTAR    296
GASEOSAS                                   270
GASEOSAS-ISOTONICA-JUGOS LIGEROS           268
GASEOSAS-ISOTONICA-JUGOS LIGEROS-AGUA       56
Name: id_cliente, dtype: int64

**Conteo de SKUs recomendadas**

In [218]:
count2 = (
    final_rec.groupby("id_cliente")["sku"]
    .apply(lambda x: "-".join(x.astype(str)))
    .reset_index(name="marcas_ordenadas")
)
count2 = (
    count2.groupby("marcas_ordenadas")["id_cliente"]
    .nunique()
    .sort_values(ascending=False)
    .reset_index()
)
count2["f"] = count2["id_cliente"] / count2["id_cliente"].sum()
count2["fa"] = count2["f"].cumsum()

In [219]:
count2

,marcas_ordenadas,id_cliente,f,fa
0,523078,45,0.033482,0.033482
1,523075,41,0.030506,0.063988
2,523075-515517-509728-514404,24,0.017857,0.081845
3,500147-515517,23,0.017113,0.098958
4,523080,23,0.017113,0.116071
...,...,...,...,...
536,500187-515417-523578,1,0.000744,0.997024
537,523078-524379,1,0.000744,0.997768
538,523078-523578-508514-599479,1,0.000744,0.998512
539,523078-523578,1,0.000744,0.999256


In [220]:
print(f"RESUMEN para {fecha_tomorrow}")
print(
    "Total de clientes a recomendar: ",
    recomendaciones_para_salesforce[["Compania","Cliente"]].drop_duplicates().shape[0],
)
print("Combinaciones de recomendaciones de MARCAS:", count.shape[0])
print("Marcar recomendadas:", final_rec.marca_rec.unique())
print("Combinaciones de recomendaciones de SKUs:", count2.shape[0])
print(
    "Combinaciones de recomendaciones de SKUs al 80% de clientes:",
    count2[count2["fa"] <= 0.8].shape[0],
)
print("SKUs usados en la recomendacion:", final_rec["sku"].nunique())

RESUMEN para 2026-04-01
Total de clientes a recomendar:  1344
Combinaciones de recomendaciones de MARCAS: 29
Marcar recomendadas: ['GASEOSAS' 'JUGOS LIGEROS' 'NECTAR' 'ISOTONICA' 'ENERGIZANTE' 'AGUA']
Combinaciones de recomendaciones de SKUs: 541
Combinaciones de recomendaciones de SKUs al 80% de clientes: 272
SKUs usados en la recomendacion: 49


In [221]:
recomendaciones_para_salesforce['tipoRecomendacion'] = recomendaciones_para_salesforce.groupby(['Pais', 'Compania', 'Sucursal', 'Cliente'])\
               .cumcount()\
               .apply(lambda x: f'PS{x+1}')

In [222]:
recomendaciones_para_salesforce["ultFecha"] = ''
recomendaciones_para_salesforce["Destacar"] = "true"

In [223]:
recomendaciones_para_salesforce

,Pais,Compania,Sucursal,Cliente,Modulo,Producto,Cajas,Unidades,Fecha,tipoRecomendacion,ultFecha,Destacar
0,EC,6003,01,1398,80312,523075,1,0,2026-04-01,PS1,,true
1,EC,6003,01,1398,80312,509728,1,0,2026-04-01,PS2,,true
2,EC,6003,01,1398,80312,514422,1,0,2026-04-01,PS3,,true
3,EC,6003,01,1399,80312,523075,1,0,2026-04-01,PS1,,true
4,EC,6003,01,1399,80312,515417,1,0,2026-04-01,PS2,,true
...,...,...,...,...,...,...,...,...,...,...,...,...
3468,EC,6010,01,997,81016,515517,1,0,2026-04-01,PS2,,true
3469,EC,6010,01,999,81013,523104,1,0,2026-04-01,PS1,,true
3470,EC,6010,01,999,81013,515904,1,0,2026-04-01,PS2,,true
3471,EC,6010,01,999,81013,509728,1,0,2026-04-01,PS3,,true


In [224]:
# eco sambo -> 6010
# eco duran -> 6003
# eco pedro carbo -> 6007
# companias_piloto_test = ['6003','6007','6010']
# recomendaciones_para_salesforce = recomendaciones_para_salesforce[recomendaciones_para_salesforce.Compania.isin(companias_piloto_test)].reset_index(drop=True)

In [225]:
recomendaciones_para_salesforce[["Compania","Cliente"]].drop_duplicates().shape

(1344, 2)

In [226]:
recomendaciones_para_salesforce.groupby(["Compania","Cliente"])["Producto"].nunique().mean()

2.584077380952381

In [227]:
recomendaciones_para_salesforce.to_csv(
    f"Output/PS_piloto_v1/D_base_pedidos_{fecha_tomorrow}.csv", index=False
)

In [228]:
# Establecer la conexión con S3
bucket_name = 'aje-analytics-ps-backup'  # nombre de bucket en S3
file_name = f'Econoredes/Ecuador/Output/PS_piloto_v1/D_base_pedidos_{fecha_tomorrow}.csv'  # nombre para el archivo en S3
s3_path = f's3://{bucket_name}/{file_name}'

# Escribir el dataframe en S3 con AWS Data Wrangler
wr.s3.to_csv(recomendaciones_para_salesforce, s3_path, index=False)

{'paths': ['s3://aje-analytics-ps-backup/Econoredes/Ecuador/Output/PS_piloto_v1/D_base_pedidos_2026-04-01.csv'],
 'partitions_values': {}}

In [229]:
# SUBE LA INFO A SALESFORCE

# Establecer la conexión con S3
bucket_name = 'aje-prd-pedido-sugerido-orders-s3'  # nombre de bucket en S3
file_name = f'PE/pedidos/base_pedidos.csv'  # nombre para el archivo en S3
s3_path = f's3://{bucket_name}/{file_name}'

# Escribir el dataframe en S3 con AWS Data Wrangler
wr.s3.to_csv(recomendaciones_para_salesforce, s3_path, index=False)

{'paths': ['s3://aje-prd-pedido-sugerido-orders-s3/PE/pedidos/base_pedidos.csv'],
 'partitions_values': {}}

In [230]:
# Apaga el Kernel para luego poder usar los demás notebooks sin llenar la memoria
import IPython
app = IPython.get_ipython()
app.kernel.do_shutdown(False)  # True indica apagar completamente el kernel


{'status': 'ok', 'restart': False}

In [231]:
#f